[Back to Data Structures and Algorithms guideline](Data-Structure&Algorithm.html)


## **Dynamic Programming** {#dynamic-programming}

**Dynamic programming (DP)** solves a problem by identifying repeated subproblems, solving each distinct subproblem once, and reusing its answer wherever that state appears again. The stored answer may be a minimum cost, maximum value, number of constructions, feasibility decision, probability, or an actual witness. DP is therefore a way to organize dependencies, not merely a way to fill a table.

The central design question is: **what information from the past can still affect the future?** A state stores exactly that information. Two decision histories that produce the same future-relevant state can be merged, because every continuation available to one is also available to the other. This state merging is what can turn an exponential recursion tree into a polynomial-size directed acyclic graph.

Dynamic programming sits between the surrounding design paradigms:

- a greedy algorithm proves that one local choice is safe and discards all alternatives;
- backtracking preserves distinct choice histories and explores them when necessary;
- dynamic programming preserves alternatives but merges histories that lead to the same state.

This chapter moves from recognition and state design to common state shapes: one-dimensional prefixes, capacities, pairs of prefixes, grids, and intervals. It then separates computing the optimal value from reconstructing the choices that achieve it.

::: {.callout-important}
Do not begin by asking, "What should the DP array contain?" Begin by writing a complete sentence: "State X means ..." The recurrence, base cases, evaluation order, and final answer should all follow from that sentence.
:::


### **Recognizing Dynamic Programming Problems** {#recognizing-dynamic-programming-problems}

DP becomes a strong candidate when a direct recursive formulation has two structural properties.

**Optimal substructure** means that an optimal answer can be assembled from optimal answers to smaller states. For counting and decision problems, the analogous requirement is that the current answer can be assembled completely from smaller answers. This property explains why smaller results are sufficient.

**Overlapping subproblems** means that different decision paths request the same state. Without caching, that state is recomputed repeatedly. A divide-and-conquer algorithm also solves smaller problems, but its recursive branches are usually independent; DP is especially valuable when the branches converge.

Useful recognition signals include:

- the problem asks for a minimum, maximum, count, or existence over many combinations;
- a choice divides the problem into a small number of smaller states;
- the same suffix, prefix, capacity, position, mask, or interval appears under many histories;
- a brute-force recursion is easy to describe but repeats calls;
- the future depends on a compact summary rather than the entire construction history.

The generic top-down structure is:

~~~text
SOLVE(state s)
    if s is a base state
        return its direct answer
    if s is already stored
        return stored[s]

    answer <- combine SOLVE(next_state) over every legal final/next decision
    stored[s] <- answer
    return answer
~~~

![Fibonacci dependencies form a DAG because several recursive paths request the same smaller state.](assets/dp-overlapping-subproblems.svg){fig-align="center" width="34%"}

*Visual source: [Fibonacci dynamic programming](https://commons.wikimedia.org/wiki/File:Fibonacci_dynamic_programming.svg), public domain. The unchanged SVG is stored locally for reliable rendering.*

Fibonacci numbers are a deliberately small example. In the naive recursion, both $F(n-1)$ and $F(n-2)$ recursively request states such as $F(n-3)$. Memoization changes the computation from a tree of calls into one evaluation per integer state. The same transformation applies to much richer states such as <code>(row, column)</code>, <code>(item, capacity)</code>, or <code>(left, right)</code>.

<details>
<summary>Python implementation: expose repeated calls and remove them with caching</summary>

~~~python
from functools import cache


def compare_fibonacci_calls(n: int) -> tuple[int, int, int]:
    """Return Fibonacci(n), naive body calls, and memoized body calls."""
    if n < 0:
        raise ValueError("n must be non-negative")

    naive_calls = 0

    def naive(k: int) -> int:
        nonlocal naive_calls
        naive_calls += 1
        if k < 2:
            return k
        return naive(k - 1) + naive(k - 2)

    memo_calls = 0

    @cache
    def memoized(k: int) -> int:
        nonlocal memo_calls
        memo_calls += 1  # The body runs only on a cache miss.
        if k < 2:
            return k
        return memoized(k - 1) + memoized(k - 2)

    naive_value = naive(n)
    memoized_value = memoized(n)
    assert naive_value == memoized_value
    return naive_value, naive_calls, memo_calls


value, naive_count, memo_count = compare_fibonacci_calls(10)
assert (value, naive_count, memo_count) == (55, 177, 11)
print(value, naive_count, memo_count)
~~~

</details>

The naive recursion takes $\Theta(\varphi^n)$ time, where $\varphi$ is the golden ratio, because its call tree grows exponentially. Memoization evaluates the $n+1$ states $0$ through $n$ once, taking $O(n)$ time and $O(n)$ cache plus recursion-stack space.

Recognition is only the first step. An inefficient or incomplete state can still produce an exponential number of distinct entries. The remaining sections focus on making the state both sufficient and economical.

**Practice:** [LeetCode 70 - Climbing Stairs](https://leetcode.com/problems/climbing-stairs/) is a compact exercise in identifying repeated suffix lengths and converting a recurrence into linear work.


### **State, Transition, Base Case, and Evaluation Order** {#state-transition-base-case-and-evaluation-order}

A DP formulation has four inseparable parts.

1. **State meaning:** a precise subproblem definition, including index conventions.
2. **Transition:** a complete partition of the legal final or next decisions.
3. **Base cases:** smallest states whose answers are known directly.
4. **Evaluation order:** an order in which every dependency is available before use.

Consider maximum non-adjacent sum. Let $v_0,\ldots,v_{n-1}$ be non-negative values, and define $dp[i]$ as the maximum sum obtainable from the **first $i$ values** without selecting adjacent positions. The prefix convention matters: $dp[0]$ describes an empty prefix, while value $v_{i-1}$ is the last value inside state $dp[i]$.

Every optimal solution for the first $i$ values makes exactly one of two final decisions:

- skip $v_{i-1}$, leaving the best answer $dp[i-1]$;
- take $v_{i-1}$, which forbids $v_{i-2}$ and leaves $v_{i-1}+dp[i-2]$.

Therefore,

$$
dp[i]=\max\left(dp[i-1],\;v_{i-1}+dp[i-2]\right),\qquad i\ge 2,
$$

with $dp[0]=0$ and $dp[1]=v_0$. Here $i$ is a prefix length, $v_{i-1}$ is the final available value, and the two arguments of <code>max</code> represent the exhaustive skip/take cases. Because state $i$ depends only on smaller indices, increasing $i$ is a valid topological order.

~~~text
MAX-NON-ADJACENT(values v[0..n-1])
    if n = 0
        return 0
    dp[0] <- 0
    dp[1] <- v[0]
    for i <- 2 to n
        skip <- dp[i-1]
        take <- v[i-1] + dp[i-2]
        dp[i] <- max(skip, take)
    return dp[n]
~~~

![A complete DP formulation connects state meaning, transition, base cases, and dependency order.](assets/dp-state-transition.svg){fig-align="center" width="100%"}

Correctness follows by induction on $i$. Assume all smaller prefix states are optimal. Every feasible answer either excludes or includes the final value; the recurrence evaluates the optimal result in both disjoint cases and chooses the better one. The base states anchor the induction.

<details>
<summary>Python implementation: maximum non-adjacent sum with an explicit state table</summary>

~~~python
def maximum_non_adjacent_sum(values: list[int]) -> int:
    """Return the maximum sum of non-adjacent non-negative values."""
    if any(value < 0 for value in values):
        raise ValueError("this state definition assumes non-negative values")
    if not values:
        return 0

    # dp[i] means the best answer using values[:i].
    dp = [0] * (len(values) + 1)
    dp[1] = values[0]

    for i in range(2, len(values) + 1):
        skip = dp[i - 1]
        take = values[i - 1] + dp[i - 2]
        dp[i] = max(skip, take)

    return dp[-1]


assert maximum_non_adjacent_sum([2, 7, 9, 3, 1]) == 12
assert maximum_non_adjacent_sum([]) == 0
print(maximum_non_adjacent_sum([2, 7, 9, 3, 1]))
~~~

</details>

There are $n+1$ states and $O(1)$ work per state, so the algorithm takes $O(n)$ time and $O(n)$ table space. The dependency window has width two, so the value alone can later be compressed to $O(1)$ space. That optimization should be applied only after the full state and order are correct.

**Practice:** [LeetCode 198 - House Robber](https://leetcode.com/problems/house-robber/) directly tests the skip/take partition and prefix-state convention.


### **Memoization and Tabulation** {#memoization-and-tabulation}

**Memoization** and **tabulation** evaluate the same state graph in different ways.

Memoization is top-down. The program starts from the requested target, follows recursive dependencies, and stores each completed answer in a cache. It often mirrors the mathematical recurrence and may avoid unreachable states. Its costs include function calls, recursion depth, and cache-key construction.

Tabulation is bottom-up. The program explicitly chooses a dependency-respecting order and fills a table from base cases toward the target. It avoids recursion overhead, makes memory access predictable, and exposes opportunities for rolling-array optimization, but may compute states that the target never reaches.

~~~text
TOP-DOWN(state s)
    if s is a base state: return base value
    if s is cached: return cache[s]
    cache[s] <- combine TOP-DOWN(predecessor) over dependencies
    return cache[s]

BOTTOM-UP(all states)
    initialize every base state
    for state s in topological dependency order
        table[s] <- combine table[predecessor] over dependencies
    return table[target]
~~~

![Memoization follows demanded recursive edges; tabulation fills the same DAG in an explicit order.](assets/memoization-tabulation.svg){fig-align="center" width="100%"}

For minimum climbing cost, let $f(i)$ be the minimum total cost paid when landing on step $i$. The final move to $i$ comes from $i-1$ or $i-2$:

$$
f(i)=cost[i]+\min(f(i-1),f(i-2)).
$$

The top lies immediately beyond the final step, so the answer is $\min(f(n-1),f(n-2))$. The recursive version asks only for states reachable from those two targets; the iterative version computes the same values left to right.

<details>
<summary>Python implementation: the same recurrence top-down and bottom-up</summary>

~~~python
from functools import cache


def minimum_climbing_cost_top_down(cost: list[int]) -> int:
    """Pay a step's cost when landing on it; start from step 0 or 1."""
    if len(cost) < 2 or any(value < 0 for value in cost):
        raise ValueError("provide at least two non-negative step costs")

    @cache
    def reach(i: int) -> int:
        if i < 2:
            return cost[i]
        return cost[i] + min(reach(i - 1), reach(i - 2))

    return min(reach(len(cost) - 1), reach(len(cost) - 2))


def minimum_climbing_cost_bottom_up(cost: list[int]) -> int:
    if len(cost) < 2 or any(value < 0 for value in cost):
        raise ValueError("provide at least two non-negative step costs")

    dp = cost.copy()
    for i in range(2, len(dp)):
        dp[i] += min(dp[i - 1], dp[i - 2])
    return min(dp[-1], dp[-2])


example = [10, 15, 20]
assert minimum_climbing_cost_top_down(example) == 15
assert minimum_climbing_cost_bottom_up(example) == 15
print(minimum_climbing_cost_top_down(example))
print(minimum_climbing_cost_bottom_up(example))
~~~

</details>

Both implementations have $n$ distinct states and $O(1)$ work per state, giving $O(n)$ time. Top-down uses $O(n)$ cache and recursion-stack space. The shown bottom-up table uses $O(n)$ space and can be reduced to two variables. On a sparse state graph, memoization may evaluate fewer states; on a dense regular table, tabulation is often simpler and faster in practice.

| Question | Memoization | Tabulation |
|---|---|---|
| Starting point | requested target | base states |
| Order | generated by recursion | written explicitly |
| Unreachable states | usually skipped | may be filled |
| Stack overflow risk | possible | absent |
| Space compression | less direct | often straightforward |

**Practice:** [LeetCode 746 - Min Cost Climbing Stairs](https://leetcode.com/problems/min-cost-climbing-stairs/) lets you implement both evaluation styles over the same recurrence.


### **One-Dimensional Dynamic Programming** {#one-dimensional-dynamic-programming}

One-dimensional DP usually processes a sequence, amount, or ordered set of positions. A common state is <code>dp[i]</code>, an answer for the first $i$ elements or for numeric target $i$. This shape is effective when every transition depends on a bounded number of earlier positions or on earlier amounts.

The important decision is not the array dimension but the **meaning of the index**. Prefix-length indexing often simplifies base cases because <code>dp[0]</code> naturally represents the empty prefix.

Consider decoding a digit string where <code>1</code> through <code>26</code> map to letters. Define $dp[i]$ as the number of valid decodings of the first $i$ characters. The empty prefix has one neutral decoding, so $dp[0]=1$. For each $i$:

- if the one-digit token $s[i-1]$ is between 1 and 9, append it to every decoding counted by $dp[i-1]$;
- if the two-digit token $s[i-2:i]$ is between 10 and 26, append it to every decoding counted by $dp[i-2]$.

Thus,

$$
dp[i]=\mathbf{1}_{1\le s[i-1]\le 9}\,dp[i-1]
+\mathbf{1}_{10\le s[i-2:i]\le 26}\,dp[i-2].
$$

$\mathbf{1}_{condition}$ is 1 when its condition is true and 0 otherwise. The two terms count disjoint constructions according to the length of the final token. A zero cannot stand alone, but it can participate in <code>10</code> or <code>20</code>.

~~~text
COUNT-DECODINGS(string s)
    dp[0] <- 1
    for i <- 1 to length(s)
        dp[i] <- 0
        if s[i-1] is a valid one-digit code
            dp[i] <- dp[i] + dp[i-1]
        if i >= 2 and s[i-2:i] is a valid two-digit code
            dp[i] <- dp[i] + dp[i-2]
    return dp[length(s)]
~~~

![For 226, each prefix state receives contributions from valid one-digit and two-digit final tokens.](assets/one-dimensional-dp.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: count valid decodings</summary>

~~~python
def count_decodings(digits: str) -> int:
    """Return the number of decodings under the mapping 1..26 -> A..Z."""
    if not digits:
        return 0
    if not digits.isdigit():
        raise ValueError("digits must contain decimal characters only")

    dp = [0] * (len(digits) + 1)
    dp[0] = 1  # One way to decode the empty prefix.

    for i in range(1, len(digits) + 1):
        if digits[i - 1] != "0":
            dp[i] += dp[i - 1]

        if i >= 2 and 10 <= int(digits[i - 2 : i]) <= 26:
            dp[i] += dp[i - 2]

    return dp[-1]


assert count_decodings("226") == 3
assert count_decodings("06") == 0
assert count_decodings("2101") == 1
print(count_decodings("226"))
~~~

</details>

There are $n+1$ prefix states, each with at most two transitions, so time is $O(n)$ and the displayed table uses $O(n)$ space. Because state $i$ depends only on $i-1$ and $i-2$, the count can be compressed to $O(1)$ space after correctness is established.

Other one-dimensional patterns include minimum cost to reach a position, number of ways to form an amount, longest valid prefix, and finite-state parsing. The shared skill is deciding which earlier states can legally become the current state.

**Practice:** [LeetCode 91 - Decode Ways](https://leetcode.com/problems/decode-ways/) directly exercises prefix meaning, zero handling, and multiple predecessor states.


### **Knapsack Problems** {#knapsack-problems}

Knapsack DP models decisions that consume a limited resource. In **0/1 knapsack**, each item may be used at most once. Item $i$ has weight $w_i$, value $v_i$, and the total selected weight must not exceed capacity $C$.

Define $dp[i][c]$ as the maximum value obtainable using the first $i$ items with capacity at most $c$. The final decision either excludes item $i$ or includes it when it fits:

$$
dp[i][c]=
\begin{cases}
dp[i-1][c], & w_i>c,\\
\max\left(dp[i-1][c],\;dp[i-1][c-w_i]+v_i\right), & w_i\le c.
\end{cases}
$$

$i$ is the number of available items, $c$ is the current capacity limit, and both alternatives use row $i-1$ because item $i$ cannot be reused. Base states $dp[0][c]=0$ and $dp[i][0]=0$ represent no items or no capacity.

The item dimension can be compressed into one capacity array. The capacity loop must then run **downward**: before processing item $i$, every <code>dp[c-w]</code> must still represent the previous item set. An upward loop would read values already updated by the same item and silently change the problem into unbounded knapsack.

~~~text
ZERO-ONE-KNAPSACK(items, capacity C)
    dp[0..C] <- 0
    for each item (weight w, value v)
        for c <- C down to w
            dp[c] <- max(dp[c], dp[c-w] + v)
    return dp[C]
~~~

![The 0/1 knapsack table compares excluding and including each item across capacities.](assets/knapsack-dp-animation.gif){fig-align="center" width="100%"}

*Visual source: [RDSEED, Knapsack problem dynamic programming](https://commons.wikimedia.org/wiki/File:Knapsack_problem_dynamic_programming.gif), CC BY-SA 4.0. The unchanged animation is stored locally for reliable rendering.*

<details>
<summary>Python implementation: space-optimized 0/1 knapsack value</summary>

~~~python
def zero_one_knapsack_value(
    items: list[tuple[int, int]], capacity: int
) -> int:
    """Return maximum value when each (weight, value) item is used at most once."""
    if capacity < 0:
        raise ValueError("capacity must be non-negative")
    if any(weight <= 0 or value < 0 for weight, value in items):
        raise ValueError("weights must be positive and values non-negative")

    dp = [0] * (capacity + 1)

    for weight, value in items:
        # Descending capacity preserves the previous-item layer.
        for current_capacity in range(capacity, weight - 1, -1):
            dp[current_capacity] = max(
                dp[current_capacity],
                dp[current_capacity - weight] + value,
            )

    return dp[capacity]


example_items = [(10, 60), (20, 100), (30, 120)]
assert zero_one_knapsack_value(example_items, 50) == 220
print(zero_one_knapsack_value(example_items, 50))
~~~

</details>

For $n$ items and capacity $C$, the algorithm takes $O(nC)$ time and $O(C)$ space. This is **pseudo-polynomial**: the cost is polynomial in the numeric capacity, not in the number of bits needed to encode $C$. A capacity of one billion is therefore impractical even if the input contains few items.

| Variant | Item reuse | Capacity-loop direction in 1D | Typical transition |
|---|---:|---|---|
| 0/1 knapsack | at most once | descending | previous layer only |
| Unbounded knapsack | unlimited | ascending | current layer may reuse item |
| Bounded knapsack | limited count | decomposition or count state | track remaining multiplicity |
| Subset sum | at most once | descending | boolean reachability |

**Practice:** [LeetCode 416 - Partition Equal Subset Sum](https://leetcode.com/problems/partition-equal-subset-sum/) reframes equal partitioning as 0/1 subset-sum feasibility.


### **Subsequence Dynamic Programming** {#subsequence-dynamic-programming}

A **subsequence** preserves relative order but may delete elements; a substring or subarray must remain contiguous. Subsequence problems often compare prefixes of one or two sequences because the final characters create a small, complete set of cases.

For strings $x$ and $y$, define $dp[i][j]$ as the length of the longest common subsequence (LCS) between prefixes $x[:i]$ and $y[:j]$. If the final characters match, they can extend an optimal solution for both shorter prefixes. If they differ, at least one final character is absent from an optimal common subsequence:

$$
dp[i][j]=
\begin{cases}
dp[i-1][j-1]+1, & x_{i-1}=y_{j-1},\\
\max(dp[i-1][j],dp[i][j-1]), & x_{i-1}\ne y_{j-1}.
\end{cases}
$$

$i$ and $j$ are prefix lengths. The diagonal state removes both final characters, while the upper and left states remove one. Row 0 and column 0 are zero because an empty sequence has no nonempty common subsequence.

~~~text
LCS-LENGTH(x, y)
    initialize dp[0][j] and dp[i][0] to 0
    for i <- 1 to length(x)
        for j <- 1 to length(y)
            if x[i-1] = y[j-1]
                dp[i][j] <- dp[i-1][j-1] + 1
            else
                dp[i][j] <- max(dp[i-1][j], dp[i][j-1])
    return dp[length(x)][length(y)]
~~~

![Sequence alignment fills a two-dimensional prefix matrix and traces optimal predecessor choices.](assets/sequence-alignment-dp.png){fig-align="center" width="62%"}

*Visual source: [Slowkow, Needleman-Wunsch pairwise sequence alignment](https://commons.wikimedia.org/wiki/File:Needleman-Wunsch_pairwise_sequence_alignment.png), CC0. Needleman-Wunsch uses match, mismatch, and gap scores, but its prefix-state geometry is the same family as LCS and edit-distance DP.*

<details>
<summary>Python implementation: longest common subsequence length</summary>

~~~python
def longest_common_subsequence_length(first: str, second: str) -> int:
    """Return the LCS length using a full prefix table."""
    rows = len(first) + 1
    columns = len(second) + 1
    dp = [[0] * columns for _ in range(rows)]

    for i in range(1, rows):
        for j in range(1, columns):
            if first[i - 1] == second[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    return dp[-1][-1]


assert longest_common_subsequence_length("abcde", "ace") == 3
assert longest_common_subsequence_length("abc", "def") == 0
print(longest_common_subsequence_length("abcde", "ace"))
~~~

</details>

For lengths $m$ and $n$, the table has $(m+1)(n+1)$ states and $O(1)$ work per state, giving $O(mn)$ time and $O(mn)$ space. If only the length is needed, one row can be rolled to $O(\min(m,n))$ space. Recovering an actual subsequence requires additional decisions or recomputation, addressed later.

Related formulations include edit distance, sequence alignment, shortest common supersequence, and $O(n^2)$ longest increasing subsequence. They differ in transition semantics, not merely in table dimensions.

**Practice:** [LeetCode 1143 - Longest Common Subsequence](https://leetcode.com/problems/longest-common-subsequence/) directly tests two-prefix state design and diagonal/upper/left transitions.


### **Grid and Multidimensional Dynamic Programming** {#grid-and-multidimensional-dynamic-programming}

Grid DP uses coordinates as state dimensions. A state such as $dp[r][c]$ may represent the number of paths to cell $(r,c)$, the minimum cost of reaching it, or the best score obtainable after processing that position. The movement rules determine predecessor cells and therefore the valid evaluation order.

If movement is restricted to right and down, every path into $(r,c)$ ends at either $(r-1,c)$ or $(r,c-1)$. The path-count recurrence is

$$
ways[r][c]=ways[r-1][c]+ways[r][c-1].
$$

$r$ and $c$ identify the current row and column. The first term counts paths whose final move is down; the second counts paths whose final move is right. These sets are disjoint and exhaustive. Obstacles set selected states to zero rather than changing the dependency pattern.

For minimum path sum over cell costs $g[r][c]$, replace addition of counts with minimization of predecessor costs:

$$
cost[r][c]=g[r][c]+\min(cost[r-1][c],cost[r][c-1]).
$$

~~~text
MINIMUM-GRID-PATH(grid g with R rows and C columns)
    dp[0] <- 0
    dp[1..C] <- infinity
    for r <- 0 to R-1
        for c <- 0 to C-1
            from_above <- dp[c]
            from_left <- dp[c-1] when c > 0, otherwise infinity
            dp[c] <- g[r][c] + min(from_above, from_left)
    return dp[C-1]
~~~

![Numbers on the grid count paths from the origin when each state receives transitions from above and left.](assets/grid-path-counts.svg){fig-align="center" width="58%"}

*Visual source: [Pascal's triangle pathways](https://commons.wikimedia.org/wiki/File:Pascal%27s_triangle_pathways.svg), public domain. The same dependency pattern supports path counts and minimum path costs.*

<details>
<summary>Python implementation: minimum path sum with one rolling row</summary>

~~~python
from math import inf


def minimum_grid_path_sum(grid: list[list[int]]) -> int:
    """Move only right/down from top-left to bottom-right."""
    if not grid or not grid[0]:
        raise ValueError("grid must be non-empty")
    columns = len(grid[0])
    if any(len(row) != columns for row in grid):
        raise ValueError("grid must be rectangular")

    dp = [inf] * columns

    for row_index, row in enumerate(grid):
        for column_index, cell_cost in enumerate(row):
            if row_index == 0 and column_index == 0:
                dp[0] = cell_cost
                continue

            from_above = dp[column_index]
            from_left = dp[column_index - 1] if column_index > 0 else inf
            dp[column_index] = cell_cost + min(from_above, from_left)

    return int(dp[-1])


example_grid = [[1, 3, 1], [1, 5, 1], [4, 2, 1]]
assert minimum_grid_path_sum(example_grid) == 7
print(minimum_grid_path_sum(example_grid))
~~~

</details>

For $R$ rows and $C$ columns, every cell is processed once, so time is $O(RC)$. The rolling row uses $O(C)$ space because the current state needs only the value above, still stored at <code>dp[c]</code>, and the value to the left, already updated at <code>dp[c-1]</code>.

Additional dimensions may encode remaining resources, direction, parity, visited masks, or time. Their product determines the number of states, so a "polynomial" recurrence can still be impractical if unnecessary information is placed in the state.

**Practice:** [LeetCode 64 - Minimum Path Sum](https://leetcode.com/problems/minimum-path-sum/) directly exercises grid dependencies and rolling-row evaluation.


### **Interval Dynamic Programming** {#interval-dynamic-programming}

Interval DP uses a state $dp[i][j]$ for a contiguous range from index $i$ through $j$. It is appropriate when the final operation splits, removes, or encloses an interval and leaves smaller intervals whose answers can be combined.

Unlike prefix DP, row-major order is usually invalid: $dp[i][j]$ may depend on states with both larger starting indices and smaller ending indices. The safe order is increasing interval length. All length-1 states are base cases, then length 2, and so on.

Matrix-chain multiplication is the classic example. Matrices $A_1,\ldots,A_n$ have dimensions $p_0\times p_1,p_1\times p_2,\ldots,p_{n-1}\times p_n$. Matrix multiplication is associative, but parenthesizations perform different numbers of scalar multiplications.

Let $dp[i][j]$ be the minimum cost to multiply $A_i\cdots A_j$. If the final multiplication splits after $A_k$, the left result has dimensions $p_{i-1}\times p_k$ and the right result has dimensions $p_k\times p_j$:

$$
dp[i][j]=\min_{i\le k<j}\left(dp[i][k]+dp[k+1][j]+p_{i-1}p_kp_j\right).
$$

$k$ enumerates every possible final split. The first two terms optimally construct the child products, and $p_{i-1}p_kp_j$ is the scalar cost of multiplying those two resulting matrices.

~~~text
MATRIX-CHAIN(dimensions p[0..n])
    dp[i][i] <- 0 for every matrix i
    for length <- 2 to n
        for each interval [i, j] of this length
            dp[i][j] <- infinity
            for k <- i to j-1
                candidate <- dp[i][k] + dp[k+1][j] + p[i-1]*p[k]*p[j]
                dp[i][j] <- min(dp[i][j], candidate)
    return dp[1][n]
~~~

![Interval states occupy an upper-triangular table and are filled by increasing length; each split combines two shorter intervals.](assets/interval-dp-order.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: minimum matrix-chain multiplication cost</summary>

~~~python
from math import inf


def matrix_chain_minimum_cost(dimensions: list[int]) -> int:
    """Return minimum scalar multiplications for the implied matrix chain."""
    if len(dimensions) < 2 or any(size <= 0 for size in dimensions):
        raise ValueError("dimensions must contain positive matrix dimensions")

    matrix_count = len(dimensions) - 1
    dp = [[0] * matrix_count for _ in range(matrix_count)]

    for length in range(2, matrix_count + 1):
        for left in range(0, matrix_count - length + 1):
            right = left + length - 1
            dp[left][right] = inf

            for split in range(left, right):
                combine_cost = (
                    dimensions[left]
                    * dimensions[split + 1]
                    * dimensions[right + 1]
                )
                candidate = (
                    dp[left][split]
                    + dp[split + 1][right]
                    + combine_cost
                )
                dp[left][right] = min(dp[left][right], candidate)

    return int(dp[0][matrix_count - 1]) if matrix_count else 0


assert matrix_chain_minimum_cost([10, 30, 5, 60]) == 4500
assert matrix_chain_minimum_cost([10, 20]) == 0
print(matrix_chain_minimum_cost([10, 30, 5, 60]))
~~~

</details>

There are $O(n^2)$ intervals and up to $O(n)$ split points per interval, giving $O(n^3)$ time and $O(n^2)$ space. The cubic bound arises from the three independent choices <code>left</code>, <code>right</code>, and <code>split</code>, not from using a two-dimensional array alone.

Other interval formulations include palindrome problems, optimal binary search trees, polygon triangulation, and removing elements whose reward depends on their surviving neighbors.

**Practice:** [LeetCode 312 - Burst Balloons](https://leetcode.com/problems/burst-balloons/) becomes an interval DP after reversing the perspective from "which balloon is first" to "which balloon is last inside this interval."


### **Solution Reconstruction and Space Optimization** {#solution-reconstruction-and-space-optimization}

An optimization DP can have two different output contracts:

- return only the optimal objective value;
- return a **witness**: the chosen items, path, subsequence, parenthesization, or edit operations.

The value table does not always identify the witness uniquely. Reconstruction usually stores a parent decision for each state or compares neighboring values while walking backward. If several predecessors tie, any consistent tie rule returns one optimal witness; enumerating all optimal witnesses is a different and potentially much larger problem.

Space optimization exploits the width of the dependency window. If row $i$ depends only on row $i-1$, older rows can be overwritten. However, those rows may contain the predecessor information required for reconstruction. The correct optimization depends on the output contract.

~~~text
RECONSTRUCT-LCS(x, y, table dp)
    i <- length(x), j <- length(y), answer <- empty
    while i > 0 and j > 0
        if x[i-1] = y[j-1]
            append x[i-1] to answer
            i <- i-1, j <- j-1
        else if dp[i-1][j] >= dp[i][j-1]
            i <- i-1
        else
            j <- j-1
    reverse answer
    return answer
~~~

![A full table supports parent-pointer backtracking; rolling rows preserve objective values but may discard the path.](assets/dp-reconstruction-space.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: compute and reconstruct one longest common subsequence</summary>

~~~python
def longest_common_subsequence(first: str, second: str) -> str:
    """Return one LCS; ties are resolved by moving upward first."""
    rows = len(first) + 1
    columns = len(second) + 1
    dp = [[0] * columns for _ in range(rows)]

    for i in range(1, rows):
        for j in range(1, columns):
            if first[i - 1] == second[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    # Walk backward from the target state through an optimal predecessor.
    i, j = len(first), len(second)
    reversed_answer: list[str] = []

    while i > 0 and j > 0:
        if first[i - 1] == second[j - 1]:
            reversed_answer.append(first[i - 1])
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1

    return "".join(reversed(reversed_answer))


def is_subsequence(candidate: str, sequence: str) -> bool:
    iterator = iter(sequence)
    return all(character in iterator for character in candidate)


answer = longest_common_subsequence("ABCBDAB", "BDCABA")
assert len(answer) == 4
assert is_subsequence(answer, "ABCBDAB")
assert is_subsequence(answer, "BDCABA")
print(answer)
~~~

</details>

The full LCS algorithm takes $O(mn)$ time and $O(mn)$ space, followed by $O(m+n)$ backtracking time. If only the length is needed, rolling rows reduce space to $O(\min(m,n))$. Hirschberg's algorithm can reconstruct an LCS in linear space by combining rolling-row computations with divide-and-conquer, trading simplicity for a more advanced reconstruction method.

Before compressing memory, ask which values remain live, whether updates need old or new values, and whether the final output requires a path. Space optimization is a dependency analysis, not a mechanical replacement of arrays with variables.

**Practice:** [LeetCode 1092 - Shortest Common Supersequence](https://leetcode.com/problems/shortest-common-supersequence/) requires an actual sequence, making reconstruction part of the problem rather than an optional afterthought.


### **Dynamic Programming, Greedy, and Backtracking** {#dynamic-programming-greedy-and-backtracking}

These techniques can begin from the same decision tree but preserve different information.

Greedy follows one path after proving each commitment safe. Backtracking keeps distinct histories because their future feasibility may differ. Dynamic programming keeps all relevant alternatives but merges histories that have the same future-relevant state and retains only the best value, count, or feasibility result for that state.

~~~text
SELECT-A-STRATEGY(problem)
    if one local choice can be proved safe
        use greedy
    else if equivalent future states recur under many histories
        define and cache/tabulate those states with DP
    else if distinct histories must remain distinct
        use backtracking or another state-space search
~~~

![Greedy commits to one path, DP merges equivalent states, and backtracking explores the choice tree.](assets/dp-strategy-comparison.svg){fig-align="center" width="100%"}

Coin change with denominations $\{1,3,4\}$ and target 6 provides a clean comparison. Largest-first greedy returns $4+1+1$ and is wrong. Backtracking finds $3+3$ but revisits the same remaining amounts through many orderings. DP stores the best result for each amount and merges those histories.

<details>
<summary>Python implementation: compare greedy, backtracking, and DP on the same state space</summary>

~~~python
from math import inf


def greedy_coin_count(coins: list[int], amount: int) -> int | None:
    count = 0
    for coin in sorted(coins, reverse=True):
        used, amount = divmod(amount, coin)
        count += used
    return count if amount == 0 else None


def backtracking_coin_count(coins: list[int], amount: int) -> int | None:
    """Exact but intentionally uncached; useful only for small examples."""
    def search(remaining: int) -> int:
        if remaining == 0:
            return 0
        best = inf
        for coin in coins:
            if coin <= remaining:
                best = min(best, 1 + search(remaining - coin))
        return best

    result = search(amount)
    return None if result == inf else int(result)


def dynamic_programming_coin_count(coins: list[int], amount: int) -> int | None:
    dp = [0] + [inf] * amount
    for current in range(1, amount + 1):
        for coin in coins:
            if coin <= current:
                dp[current] = min(dp[current], 1 + dp[current - coin])
    return None if dp[amount] == inf else int(dp[amount])


coins = [1, 3, 4]
assert greedy_coin_count(coins, 6) == 3
assert backtracking_coin_count(coins, 6) == 2
assert dynamic_programming_coin_count(coins, 6) == 2
print("greedy:", greedy_coin_count(coins, 6))
print("backtracking:", backtracking_coin_count(coins, 6))
print("dynamic programming:", dynamic_programming_coin_count(coins, 6))
~~~

</details>

For $k$ coin types and target $A$, the greedy rule takes $O(k\log k)$ time but lacks correctness for arbitrary denominations. The uncached backtracking recursion can grow exponentially with $A$. DP takes $O(kA)$ time and $O(A)$ space by solving each remaining amount once.

| Structural question | Greedy | Dynamic programming | Backtracking |
|---|---|---|---|
| Must alternatives be retained? | no, after safety proof | yes, but merge equal states | yes, keep distinct histories |
| Typical correctness tool | exchange / stays-ahead | recurrence + induction | exhaustive coverage + safe pruning |
| Repeated equivalent states | discarded by proof | cached or tabulated | often recomputed unless memoized |
| Typical worst-case scale | linear or sorting-dominated | number of states times transitions | exponential search tree |

Choose DP when you can state what makes two partial histories equivalent for every possible continuation. If that equivalence cannot be expressed compactly, backtracking may be necessary. If one alternative can be proved permanently dominant, greedy is simpler and usually faster.

**Practice:** [LeetCode 139 - Word Break](https://leetcode.com/problems/word-break/) is a strong recognition exercise: naive segmentation revisits suffixes, while a prefix-feasibility state merges equivalent histories.
